In [1]:
import tensorflow as tf
print(tf.__version__)

2.21.0


In [4]:
import os
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping

print("TensorFlow Version:", tf.__version__)

Matplotlib is building the font cache; this may take a moment.


TensorFlow Version: 2.21.0


In [7]:
train_path = "dataset/Dataset_img/train"
valid_path = "dataset/Dataset_img/valid"
test_path = "dataset/Dataset_img/test"

print("Train:", os.path.exists(train_path))
print("Validation:", os.path.exists(valid_path))
print("Test:", os.path.exists(test_path))

Train: True
Validation: True
Test: True


In [8]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

train_generator = ImageDataGenerator(
    preprocessing_function=tf.keras.applications.resnet50.preprocess_input,
    rotation_range=20,
    zoom_range=0.2,
    horizontal_flip=True
)

valid_generator = ImageDataGenerator(
    preprocessing_function=tf.keras.applications.resnet50.preprocess_input
)

test_generator = ImageDataGenerator(
    preprocessing_function=tf.keras.applications.resnet50.preprocess_input
)

In [9]:
train_data = train_generator.flow_from_directory(
    train_path,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="binary"
)

valid_data = valid_generator.flow_from_directory(
    valid_path,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="binary"
)

test_data = test_generator.flow_from_directory(
    test_path,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="binary",
    shuffle=False
)

Found 2526 images belonging to 2 classes.
Found 200 images belonging to 2 classes.
Found 200 images belonging to 2 classes.


In [10]:
print(train_data.class_indices)
print("Training Images :", train_data.samples)
print("Validation Images :", valid_data.samples)
print("Testing Images :", test_data.samples)

{'autistic': 0, 'non_autistic': 1}
Training Images : 2526
Validation Images : 200
Testing Images : 200


In [11]:
base_model = ResNet50(
    weights="imagenet",
    include_top=False,
    input_shape=(224, 224, 3)
)

# Freeze the ResNet50 layers
base_model.trainable = False

x = base_model.output

x = GlobalAveragePooling2D()(x)

x = Dropout(0.5)(x)

output = Dense(1, activation="sigmoid")(x)

model = Model(inputs=base_model.input, outputs=output)

model.summary()

94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 90s 1us/step


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_pad           │ (None, 230, 230,  │          0 │ input_layer[0][0] │
│ (ZeroPadding2D)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_conv (Conv2D) │ (None, 112, 112,  │      9,472 │ conv1_pad[0][0]   │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_bn            │ (None, 112, 112,  │        256 │ conv1_conv[0][0]  │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_relu          │ (None, 112, 112,  │          0 │ conv1_bn[0][0]    │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool1_pad           │ (None, 114, 114,  │          0 │ conv1_relu[0][0]  │
│ (ZeroPadding2D)     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool1_pool          │ (None, 56, 56,    │          0 │ pool1_pad[0][0]   │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_conv │ (None, 56, 56,    │      4,160 │ pool1_pool[0][0]  │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_bn   │ (None, 56, 56,    │        256 │ conv2_block1_1_c… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_relu │ (None, 56, 56,    │          0 │ conv2_block1_1_b… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_conv │ (None, 56, 56,    │     36,928 │ conv2_block1_1_r… │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_bn   │ (None, 56, 56,    │        256 │ conv2_block1_2_c… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_relu │ (None, 56, 56,    │          0 │ conv2_block1_2_b… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_0_conv │ (None, 56, 56,    │     16,640 │ pool1_pool[0][0]  │
│ (Conv2D)            │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_3_conv │ (None, 56, 56,    │     16,640 │ conv2_block1_2_r… │
│ (Conv2D)            │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_0_bn   │ (None, 56, 56,    │      1,024 │ conv2_block1_0_c… │
│ (BatchNormalizatio… │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_3_bn   │ (None, 56, 56,    │      1,024 │ conv2_block1_3_c

 Total params: 23,589,761 (89.99 MB)

 Trainable params: 2,049 (8.00 KB)

 Non-trainable params: 23,587,712 (89.98 MB)

In [12]:
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

In [13]:
checkpoint = ModelCheckpoint(
    "models/resnet50_best.keras",
    monitor="val_accuracy",
    save_best_only=True,
    mode="max",
    verbose=1
)

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

In [14]:
history = model.fit(
    train_data,
    validation_data=valid_data,
    epochs=10,
    callbacks=[checkpoint, early_stop]
)

Epoch 1/10
79/79 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.6346 - loss: 0.6926
Epoch 1: val_accuracy improved from None to 0.65500, saving model to models/resnet50_best.keras

Epoch 1: finished saving model to models/resnet50_best.keras
79/79 ━━━━━━━━━━━━━━━━━━━━ 314s 4s/step - accuracy: 0.6346 - loss: 0.6926 - val_accuracy: 0.6550 - val_loss: 0.5954
Epoch 2/10
79/79 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.7308 - loss: 0.5548
Epoch 2: val_accuracy improved from 0.65500 to 0.69000, saving model to models/resnet50_best.keras

Epoch 2: finished saving model to models/resnet50_best.keras
79/79 ━━━━━━━━━━━━━━━━━━━━ 257s 3s/step - accuracy: 0.7308 - loss: 0.5548 - val_accuracy: 0.6900 - val_loss: 0.5611
Epoch 3/10
79/79 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.7542 - loss: 0.5116
Epoch 3: val_accuracy improved from 0.69000 to 0.73500, saving model to models/resnet50_best.keras

Epoch 3: finished saving model to models/resnet50_best.keras
79/79 ━━━━━━━━━━━━━━━━━━━━ 271s 3s/step

In [15]:
# Unfreeze only the last 30 layers
base_model.trainable = True

for layer in base_model.layers[:-30]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

In [16]:
history_finetune = model.fit(
    train_data,
    validation_data=valid_data,
    epochs=20,
    callbacks=[checkpoint, early_stop]
)

Epoch 1/20
79/79 ━━━━━━━━━━━━━━━━━━━━ 0s 4s/step - accuracy: 0.7193 - loss: 0.6169
Epoch 1: val_accuracy did not improve from 0.77500
79/79 ━━━━━━━━━━━━━━━━━━━━ 379s 5s/step - accuracy: 0.7193 - loss: 0.6169 - val_accuracy: 0.7100 - val_loss: 0.5507
Epoch 2/20
79/79 ━━━━━━━━━━━━━━━━━━━━ 0s 4s/step - accuracy: 0.7886 - loss: 0.4668
Epoch 2: val_accuracy did not improve from 0.77500
79/79 ━━━━━━━━━━━━━━━━━━━━ 296s 4s/step - accuracy: 0.7886 - loss: 0.4668 - val_accuracy: 0.7350 - val_loss: 0.5091
Epoch 3/20
79/79 ━━━━━━━━━━━━━━━━━━━━ 0s 4s/step - accuracy: 0.7965 - loss: 0.4298
Epoch 3: val_accuracy did not improve from 0.77500
79/79 ━━━━━━━━━━━━━━━━━━━━ 302s 4s/step - accuracy: 0.7965 - loss: 0.4298 - val_accuracy: 0.7700 - val_loss: 0.4854
Epoch 4/20
79/79 ━━━━━━━━━━━━━━━━━━━━ 0s 4s/step - accuracy: 0.7997 - loss: 0.4268
Epoch 4: val_accuracy improved from 0.77500 to 0.79000, saving model to models/resnet50_best.keras

Epoch 4: finished saving model to models/resnet50_best.keras
79/79 

In [17]:
print("Training Accuracy:", history_finetune.history['accuracy'][-1])
print("Validation Accuracy:", history_finetune.history['val_accuracy'][-1])

Training Accuracy: 0.9279493093490601
Validation Accuracy: 0.8349999785423279


In [18]:
print("Best Validation Accuracy:",
      max(history_finetune.history['val_accuracy']))
      

Best Validation Accuracy: 0.8600000143051147


In [19]:
from tensorflow.keras.models import load_model

best_model = load_model("models/resnet50_best.keras")

test_loss, test_acc = best_model.evaluate(test_data)

print(f"Test Accuracy: {test_acc*100:.2f}%")

7/7 ━━━━━━━━━━━━━━━━━━━━ 20s 2s/step - accuracy: 0.9050 - loss: 0.2667
Test Accuracy: 90.50%
